In [ ]:
import os
from dotenv import load_dotenv
import gradio as gr
import matplotlib.pyplot as plt

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS       # <--- vector store
from langchain_huggingface import HuggingFaceEmbeddings  # <--- to enable embedding models
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Load API key safely from .env
load_dotenv()

## 1. Documents for the demo

In [ ]:
docs = [
    Document(page_content="LangChain is a framework for building LLM applications."),
    Document(page_content="RAG means Retrieval-Augmented Generation."),
    Document(page_content="BM25 is a keyword-based retrieval algorithm."),
    Document(page_content="Vector databases are useful for semantic search."),
    Document(page_content="Paris is the capital of France."),
    Document(page_content="Denis Molin is a great guy."),
]

## 2. Retriever / vector store

In [ ]:
embeddings  = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)

## 3. LLM

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 4. RAG function

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

SYSTEM_PROMPT = """
You are a helpful assistant answering only from the provided context.

If the answer is not in the context, say:
"I don't know based on the provided documents."

Do not invent information.
"""

def build_rag_answer(vectorstore, llm, k=4, return_sources=False):
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})

    prompt = ChatPromptTemplate.from_template("""
{system_prompt}

Context:
{context}

Question:
{question}

Answer:
""")

    def rag_answer(question: str):
        docs = retriever.invoke(question)
        context = "\n\n".join(doc.page_content for doc in docs)

        messages = prompt.format_messages(
            system_prompt=SYSTEM_PROMPT,
            context=context,
            question=question,
        )

        response = llm.invoke(messages)

        if hasattr(response, "content"):
            answer = str(response.content)
        else:
            answer = str(response)

        if not return_sources:
            return answer

        sources_md = "\n\n".join(
            f"**Source {i+1}**\n\n{doc.page_content}"
            for i, doc in enumerate(docs)
        )

        if not sources_md.strip():
            sources_md = "No sources retrieved."

        return answer, sources_md

    return rag_answer

## 5. Chat interface

In [ ]:
from rag_ui import launch_rag_ui

# Example:
# vectorstore = FAISS.from_documents(chunks, embedding_model)
# llm = ChatOpenAI(model="gpt-4o-mini")  # or your Mistral / LiteLLM wrapper

rag_answer = build_rag_answer(vectorstore=vectorstore, llm=llm, k=4,return_sources=True)

def chat(message, history):
    answer, sources_md = rag_answer(message)
    history = history + [{"role": "user", "content": message},
                         {"role": "assistant", "content": answer}]
    return history, sources_md, ""

demo = launch_rag_ui(
    chat_fn=chat,
    title="Simple RAG Demo (Vector Embeddings)",
    port=7860,
)

## 6. Embedding vectors inspection

### 6.1 Inspect the vector stor index

In [ ]:
index       = vectorstore.index
num_vectors = index.ntotal
vector_dim  = index.d

print(f"Number of vectors: {num_vectors}")
print(f"Vector dimension: {vector_dim}")

# Recover vectors stored
doc_vectors  = index.reconstruct_n(0, num_vectors)
print("Document vectors shape:", doc_vectors.shape)

### 6.2 Prepare labels

In [ ]:
labels = [doc.page_content[:40].replace("\n", " ") for doc in docs[:num_vectors]]

### 6.3 Compare two query embeddings

In [ ]:
import numpy as np
query_1 = "What is RAG?"
query_2 = "Explain retrieval augmented generation"

q1_vec = np.array(embeddings.embed_query(query_1))
q2_vec = np.array(embeddings.embed_query(query_2))

query_similarity = cosine_similarity([q1_vec], [q2_vec])[0][0]
print(f'Cosine similarity between "{query_1}" and "{query_2}": {query_similarity:.4f}')

### 6.4 Project document + query vectors into 2D with PCA

In [ ]:
all_vectors = np.vstack([doc_vectors, q1_vec, q2_vec])

pca         = PCA(n_components=2)
all_2d      = pca.fit_transform(all_vectors)

doc_2d      = all_2d[:num_vectors]
q1_2d       = all_2d[num_vectors]
q2_2d       = all_2d[num_vectors + 1]

### 6.5 Matplotlib visualization

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 7))

# Document points
plt.scatter(doc_2d[:, 0], doc_2d[:, 1], s=80, label="Documents")

# Faint arrows from origin to document vectors
for i, label in enumerate(labels):
    x, y = doc_2d[i]
    plt.arrow(
        0, 0, x, y,
        alpha=0.25,
        head_width=0.03,
        length_includes_head=True
    )
    plt.text(x, y, label, fontsize=9)

# Query points
plt.scatter(q1_2d[0], q1_2d[1], s=140, marker="X", label="Query 1")
plt.scatter(q2_2d[0], q2_2d[1], s=140, marker="X", label="Query 2")

# Query arrows
plt.arrow(
    0, 0, q1_2d[0], q1_2d[1],
    alpha=0.7,
    head_width=0.04,
    length_includes_head=True
)
plt.arrow(
    0, 0, q2_2d[0], q2_2d[1],
    alpha=0.7,
    head_width=0.04,
    length_includes_head=True
)

plt.text(q1_2d[0], q1_2d[1], f" {query_1}", fontsize=10)
plt.text(q2_2d[0], q2_2d[1], f" {query_2}", fontsize=10)

plt.axhline(0, linewidth=0.5)
plt.axvline(0, linewidth=0.5)

plt.title("Document and Query Embeddings (PCA projection)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()
plt.show()

### 6.6 Interactive Plotly visualization

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import pandas as pd

pio.renderers.default = "notebook_connected"

plot_df = pd.DataFrame({
    "x": np.concatenate([doc_2d[:, 0], [q1_2d[0], q2_2d[0]]]),
    "y": np.concatenate([doc_2d[:, 1], [q1_2d[1], q2_2d[1]]]),
    "label": labels + [query_1, query_2],
    "type": ["Document"] * num_vectors + ["Query", "Query"],
})

fig = px.scatter(
    plot_df,
    x="x",
    y="y",
    text="label",
    color="type",
    title="Interactive PCA Projection of Embeddings",
    hover_data={"x": ':.3f', "y": ':.3f', "label": True, "type": True},
)

fig.update_traces(textposition="top center")
fig.update_layout(width=900, height=650)
fig.show()

### 6.7 Retrieve top matching documents

We convert the distance into a similarity score using:

$similarity = \frac{1}{1 + distance}$

In [ ]:
query = "What is RAG?"
docs_scores = vectorstore.similarity_search_with_score(query, k=3)

print(f"Top matches for: {query}\n")

for i, (doc, distance) in enumerate(docs_scores, 1):

    similarity = 1 / (1 + distance)

    print(f"Result {i}")
    print(f"distance   = {distance:.4f}")
    print(f"similarity = {similarity:.3f}")
    print(doc.page_content)
    print("-" * 60)